**Derive Civil Date Ranges**

This notebook works out the civil dates that the gravestones do not carve and writes them into the archive's `person` table. For a person whose stone carves a Hebrew date of death but no civil one, it converts the Hebrew date with pyluach into the range of civil days it can be; for a person whose stone carves an age at death but no birth date, it estimates the range of days of birth from the age and the date of death. It reads the database named by `DATABASE_URL` (from `backend/.env`, unless the shell sets it), checks the conversion against the stones that carve both dates, and writes each range's earliest day into the date's own columns (`death_day`, `death_month`, `death_year`, and likewise for birth) and its latest day into the `_upper` columns, at the end of the run. It never changes a carved value, another table or any file, and with `APPLY = False` it writes nothing at all. `backend/import.py` applies the same rules (`backend/date_ranges.py`) on every load, so the notebook is needed only for a database loaded before it did, and running it again changes nothing. Its outputs:

1. a census of the dates on record, and the ranges derived for dates of death and of birth
2. a cross-check of the conversion against the dates carved in both calendars
3. the derived ranges, written to `person` in one transaction (rolled back while `APPLY` is False)
4. a consistency check that rebuilds every range a second way and compares it with what is stored

**1. Configuration and Helpers**

This cell takes the database address from `DATABASE_URL` (the shell's, or else the one in `../backend/.env`) and defines what every later cell reuses:

1. `APPLY`: whether section 6 commits its writes (`True`) or rolls them back (`False`, a dry run)
2. `connect` and `describe_database`: a connection to the archive, and its name and host for the reports, without the password or the host's own name
3. `hebrew_range`, `civil_range` and `birth_range`: the three date rules, which `backend/date_ranges.py` also applies, so that a load and this notebook store the same values
4. `carved_parts` and `range_columns`: a date's parts as the stone carves them, and the six columns a derived range fills

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and Helpers  —  settings, database access and the date rules
#
# Every rule the notebook applies is defined here once, so that the later
# cells only read, derive, compare and write. The rules were decided on
# 11 November 2026 and are those of backend/date_ranges.py, which import.py
# applies on every load; the two copies must agree.
#   • A Hebrew range runs from the first to the last day of its Hebrew year
#     (1 Tishrei to 29 Elul) whose day and month agree with the parts on
#     record, converted with pyluach. No day is added for a death after
#     nightfall, and a plain Adar in a leap year stays a day and a year.
#   • A carved civil date covers its whole month or year when only those are
#     carved. It is never changed, only read to estimate a birth date.
#   • The age at death counts completed years: the earliest birth is the
#     earliest death less (age + 1) years, plus a day; the latest birth is
#     the latest death less age years. 29 February becomes 28 February.
# An upper column holds a value exactly when its date was worked out.
# ─────────────────────────────────────────────────────────────────────────────
import calendar
import os
from collections import Counter
from datetime import date, timedelta
from importlib.metadata import version
from pathlib import Path

import psycopg
from dotenv import load_dotenv
from psycopg.conninfo import conninfo_to_dict
from psycopg.rows import dict_row
from pyluach import dates, hebrewcal

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
APPLY    = True                                 # False: a dry run, rolled back
REPO_DIR = Path.cwd().resolve().parent          # run from database_revisions/
ENV_FILE = REPO_DIR / "backend" / ".env"        # DATABASE_URL=postgresql://…
KINDS    = ("birth", "death")                   # a person's two civil dates
PARTS    = ("day", "month", "year")             # the parts each date is kept in
RULE     = "─" * 72                             # frames each printed report

LOWER  = {kind: tuple(f"{kind}_{part}" for part in PARTS) for kind in KINDS}
UPPER  = {kind: tuple(f"{kind}_{part}_upper" for part in PARTS) for kind in KINDS}
HEBREW = ("death_day_he", "death_month_he", "death_year_he")
DATE_COLUMNS = tuple(column for kind in KINDS for column in LOWER[kind] + UPPER[kind])


# ═════════════════════════════════════════════════════════════════════════════
# DATABASE
# ═════════════════════════════════════════════════════════════════════════════
def database_url() -> str:
    """Return DATABASE_URL from the shell, or else from backend/.env.

    Raises:
        RuntimeError: neither sets it.
    """
    load_dotenv(ENV_FILE)
    url = os.environ.get("DATABASE_URL", "").strip()
    if not url:
        raise RuntimeError(f"DATABASE_URL is set neither in the shell nor in {ENV_FILE}")
    return url


def describe_database(url: str) -> str:
    """Return a database's name and host without the password or the host's own name.

    Args:
        url: a PostgreSQL address.

    Returns:
        "neondb on ….neon.tech" for a Neon database; a local host is shown whole.
    """
    parts = conninfo_to_dict(url)
    host = parts.get("host", "localhost")
    labels = host.split(".")
    if len(labels) > 2 and not host.replace(".", "").isdigit():
        host = "…." + ".".join(labels[-2:])
    return f"{parts.get('dbname', '?')} on {host}"


def connect() -> psycopg.Connection:
    """Open a connection to the archive whose rows come back as dictionaries.

    Returns:
        An open connection; in a `with` block it commits on success and
        rolls back on an error.
    """
    return psycopg.connect(database_url(), row_factory=dict_row)


# ═════════════════════════════════════════════════════════════════════════════
# DATE RULES  —  from the parts on record to the earliest and latest day
# ═════════════════════════════════════════════════════════════════════════════
def hebrew_range(day: int | None, month: int | None,
                 year: int | None) -> tuple[date, date] | None:
    """Return the first and last civil day a Hebrew date can be.

    Args:
        day: the day of the month, or None.
        month: the month as pyluach numbers it (1 Nisan … 7 Tishrei …
            13 Adar II), or None.
        year: the Hebrew year, e.g. 5680, or None.

    Returns:
        The first and last day of the Hebrew year whose day and month agree
        with those given, converted to civil dates; None when the year is
        unknown or the calendar rules the date out.
    """
    if year is None:
        return None
    matching = [hebrew_day for hebrew_day in hebrewcal.Year(year).iterdates()
                if (day is None or hebrew_day.day == day)
                and (month is None or hebrew_day.month == month)]
    if not matching:
        return None
    return matching[0].to_pydate(), matching[-1].to_pydate()


def civil_range(day: int | None, month: int | None,
                year: int | None) -> tuple[date, date] | None:
    """Return the days a carved civil date covers: its whole month or year if only those are carved.

    Returns:
        The first and last day, or None when the year is unknown or the
        carved day does not exist.
    """
    if year is None:
        return None
    if month is None:
        return date(year, 1, 1), date(year, 12, 31)
    if day is None:
        return date(year, month, 1), date(year, month, calendar.monthrange(year, month)[1])
    try:
        return date(year, month, day), date(year, month, day)
    except ValueError:
        return None


def years_before(day: date, years: int) -> date:
    """Return the same day `years` years earlier; 29 February becomes 28 February if need be."""
    try:
        return day.replace(year=day.year - years)
    except ValueError:
        return day.replace(year=day.year - years, day=28)


def birth_range(death: tuple[date, date], age: int) -> tuple[date, date]:
    """Return the days on which someone who died within `death` aged `age` was born.

    Args:
        death: the first and last day the person may have died on.
        age: the age at death, in completed years.
    """
    return (years_before(death[0], age + 1) + timedelta(days=1),
            years_before(death[1], age))


# ═════════════════════════════════════════════════════════════════════════════
# COLUMNS
# ═════════════════════════════════════════════════════════════════════════════
def carved_parts(person: dict, kind: str) -> tuple:
    """Return the (day, month, year) of a person's `kind` date as the stone carves it.

    A date whose upper year is set was worked out by an earlier run, so its
    parts count as not carved: (None, None, None).
    """
    if person[UPPER[kind][2]] is not None:
        return None, None, None
    return tuple(person[column] for column in LOWER[kind])


def range_columns(kind: str, span: tuple[date, date]) -> dict:
    """Return the six columns a derived `kind` date fills: its first day's parts and its last's."""
    columns = {}
    for lower, upper, part in zip(LOWER[kind], UPPER[kind], PARTS):
        columns[lower] = getattr(span[0], part)
        columns[upper] = getattr(span[1], part)
    return columns


def precision(day: int | None, month: int | None) -> str:
    """Name what a date with a year gives besides it: "full date", "month and year", …."""
    if day is not None and month is not None:
        return "full date"
    if month is not None:
        return "month and year"
    return "day and year" if day is not None else "year alone"


print(f"✓ Configuration and helpers loaded (pyluach {version('pyluach')}, APPLY = {APPLY}, "
      f"{describe_database(database_url())})")

✓ Configuration and helpers loaded (pyluach 2.3.0, APPLY = True, neondb on ….neon.tech)


**2. Read the People**

This cell reads every row of `person`, joined to `grave` for each stone's reference (`W-12-4`), into `people`, and counts the rows of the other tables into `tables_before`. It stops if `person` lacks the six upper columns, which `backend/schema.sql` adds. A date whose upper year is already set counts as not carved (`carved_parts`), so that a second run derives it afresh. It reports:

1. what each stone carves of the date of death: a civil year, a Hebrew year alone, or less
2. what it carves of the birth date: a birth year, an age at death alone, or neither
3. how many ranges are stored already, from an earlier run or a load by `import.py`

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Read the People  —  every person's dates as carved, and the other tables
#
# The person table is read once. Everything later works from this copy, and
# the last cell compares the stored rows with it, so the copy is also the
# record of what the table held before this run.
#   • A date whose upper year is set was derived before; its parts are not
#     carved, and it is worked out again below.
#   • The other tables are only counted, so that the last cell can show they
#     were left alone.
# ─────────────────────────────────────────────────────────────────────────────
if "connect" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

PERSON_SQL = """
    SELECT person.*, grave.section, grave.row_label, grave.stone_number
      FROM person JOIN grave ON grave.id = person.grave_id
"""
SECTION_LETTERS = {"women": "W", "children": "C", "men": "M"}   # section → reference letter
OTHER_TABLES    = ("cemetery", "grave", "photo", "name_vectors")   # counted, never written


def missing_columns(conn: psycopg.Connection) -> list[str]:
    """Return the upper columns the person table lacks."""
    present = {row["column_name"] for row in conn.execute(
        "SELECT column_name FROM information_schema.columns WHERE table_name = 'person'")}
    return [column for kind in KINDS for column in UPPER[kind] if column not in present]


def count_tables(conn: psycopg.Connection) -> dict[str, int]:
    """Return how many rows each table other than person holds."""
    return {table: conn.execute(f"SELECT count(*) AS n FROM {table}").fetchone()["n"]
            for table in OTHER_TABLES}


def read_people(conn: psycopg.Connection) -> list[dict]:
    """Return every person row, with its grave's reference as `ref`, ordered by reference."""
    rows = conn.execute(PERSON_SQL).fetchall()
    for row in rows:
        row["ref"] = (f"{SECTION_LETTERS[row['section']]}-{row['row_label']}"
                      f"-{row['stone_number']}")
    return sorted(rows, key=lambda row: (row["ref"], str(row["id"])))


def death_on_record(person: dict) -> str:
    """Return what a person's stone carves of the date of death, as a census label."""
    day, month, year = carved_parts(person, "death")
    if year is not None:
        return "Civil year of death carved"
    if day is not None or month is not None:
        return "Civil day or month, no year"
    if person["death_year_he"] is not None:
        return "Hebrew year, no civil date"
    if person["death_day_he"] is not None or person["death_month_he"] is not None:
        return "Hebrew day or month, no year"
    return "No date of death"


def birth_on_record(person: dict) -> str:
    """Return what a person's stone carves of the birth date, as a census label."""
    day, month, year = carved_parts(person, "birth")
    if year is not None:
        return "Birth year carved"
    if day is not None or month is not None:
        return "Birth day or month, no year"
    if person["age_of_death"] is not None:
        return "Age at death, no birth date"
    return "Neither age nor birth date"


with connect() as _conn:
    _missing = missing_columns(_conn)
    if _missing:
        raise RuntimeError(f"person lacks {', '.join(_missing)}: run backend/schema.sql first "
                           "(python backend/import.py runs it before every load)")
    people = read_people(_conn)
    tables_before = count_tables(_conn)
people_by_id = {_person["id"]: _person for _person in people}

death_census = Counter(death_on_record(_person) for _person in people)
birth_census = Counter(birth_on_record(_person) for _person in people)
stored_ranges = {_kind: sum(_person[UPPER[_kind][2]] is not None for _person in people)
                 for _kind in KINDS}

print(f"{RULE}\nDates on record: person in {describe_database(database_url())}\n{RULE}")
for _title, _census in (("Date of death", death_census), ("Birth date", birth_census)):
    print(f"  {_title:<42}{'people':>8}{'share':>9}")
    for _label, _count in _census.most_common():
        print(f"    {_label:<40}{_count:>8}{_count / len(people):>9.1%}")
print(f"  {'Ranges stored already':<42}{stored_ranges['death']:>8} deaths, "
      f"{stored_ranges['birth']} births")
print(f"{RULE}\n  {len(people)} people read; "
      + ", ".join(f"{table} {count}" for table, count in tables_before.items())
      + " rows in the other tables")

────────────────────────────────────────────────────────────────────────
Dates on record: person in neondb on ….neon.tech
────────────────────────────────────────────────────────────────────────
  Date of death                               people    share
    Civil year of death carved                   564    80.2%
    Hebrew year, no civil date                    90    12.8%
    No date of death                              38     5.4%
    Hebrew day or month, no year                  10     1.4%
    Civil day or month, no year                    1     0.1%
  Birth date                                  people    share
    Age at death, no birth date                  409    58.2%
    Neither age nor birth date                   236    33.6%
    Birth year carved                             58     8.3%
  Ranges stored already                            0 deaths, 0 births
────────────────────────────────────────────────────────────────────────
  703 people read; cemetery 1, grave 794, 

**3. Derive the Death Ranges**

This cell takes `people` and, for each person whose stone carves a Hebrew year of death but no part of a civil date, converts the Hebrew date with `hebrew_range` into `death_ranges` (person id → first and last civil day). A date that matches no day of its Hebrew year (a 30th in a month that year gives 29 days) gets no range and is listed in `ruled_out`. It reports, for each precision of the Hebrew date:

1. how many people have it, and how many days their ranges span
2. how many ranges reach across a civil New Year, and so cover two civil years

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Derive the Death Ranges  —  a civil range from each Hebrew-only date
#
# Only a stone with no civil part of the date of death gets a range; a carved
# civil date, however partial, is kept as it is. The precision is what the
# Hebrew date gives besides its year:
#   • full date       → one day
#   • month and year  → the whole month (29 or 30 days)
#   • day and year    → that day of every month: a plain Adar in a leap year
#   • year alone      → the whole Hebrew year, 1 Tishrei to 29 Elul
# ─────────────────────────────────────────────────────────────────────────────
if "people" not in globals():
    raise RuntimeError("Run the 'Read the People' cell first.")

death_ranges = {}           # person id → (first day, last day) of the derived date of death
death_precision = {}        # person id → precision of the Hebrew date it came from
ruled_out = []              # references of Hebrew dates that match no day of their year

for _person in people:
    if any(part is not None for part in carved_parts(_person, "death")):
        continue
    _day, _month, _year = (_person[column] for column in HEBREW)
    if _year is None:
        continue
    _span = hebrew_range(_day, _month, _year)
    if _span is None:
        ruled_out.append(f"{_person['ref']}: {_day} / {_month} / {_year}")
        continue
    death_ranges[_person["id"]] = _span
    death_precision[_person["id"]] = precision(_day, _month)

print(f"{RULE}\nDeath ranges from the Hebrew date\n{RULE}")
print(f"  {'Hebrew date gives':<22}{'people':>8}{'days spanned':>16}{'two civil years':>18}")
for _label in ("full date", "month and year", "day and year", "year alone"):
    _spans = [death_ranges[key] for key, value in death_precision.items() if value == _label]
    if not _spans:
        continue
    _days = sorted((last - first).days + 1 for first, last in _spans)
    _two_years = sum(first.year != last.year for first, last in _spans)
    _width = f"{_days[0]}" if _days[0] == _days[-1] else f"{_days[0]}–{_days[-1]}"
    print(f"  {_label:<22}{len(_spans):>8}{_width:>16}{_two_years:>18}")
for _item in ruled_out:
    print(f"  [WARN] no day matches {_item}")
print(RULE)
if death_ranges:
    print(f"  {len(death_ranges)} dates of death worked out, from "
          f"{min(first for first, _ in death_ranges.values())} to "
          f"{max(last for _, last in death_ranges.values())}; {len(ruled_out)} ruled out")

────────────────────────────────────────────────────────────────────────
Death ranges from the Hebrew date
────────────────────────────────────────────────────────────────────────
  Hebrew date gives       people    days spanned   two civil years
  full date                   78               1                 0
  month and year               2              29                 0
  day and year                 5         355–357                 5
  year alone                   5         353–383                 5
────────────────────────────────────────────────────────────────────────
  90 dates of death worked out, from 1890-08-24 to 1972-07-16; 0 ruled out


**4. Derive the Birth Ranges**

This cell takes `people` and `death_ranges` and, for each person whose stone carves an age at death but no part of a birth date, estimates `birth_ranges` (person id → first and last civil day) with `birth_range`. The date of death it starts from is the carved civil date, as its whole month or year when only those are carved, or else the range of section 3. It reports:

1. how many ranges start from a carved and how many from a derived date of death, and how many people have no date of death to start from
2. how many ranges reach across one civil New Year or two, and how many across a decade

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Derive the Birth Ranges  —  a range of birth from the age at death
#
# An age in completed years leaves a year of possible birthdays before each
# possible day of death, so every range is about a year wide even when the
# date of death is exact:
#     earliest birth = earliest death − (age + 1) years + 1 day
#     latest birth   = latest death − age years
# A stone that carves a birth year keeps it, even where the age could narrow
# a year-only birth.
# ─────────────────────────────────────────────────────────────────────────────
if "death_ranges" not in globals():
    raise RuntimeError("Run the 'Derive the Death Ranges' cell first.")


def death_span(person: dict) -> tuple[tuple[date, date] | None, str]:
    """Return the days a person may have died on, and where they come from.

    Returns:
        The carved civil date's days and "carved", else the derived range
        and "derived", else None and "none".
    """
    carved = carved_parts(person, "death")
    if any(part is not None for part in carved):
        return civil_range(*carved), "carved"
    if person["id"] in death_ranges:
        return death_ranges[person["id"]], "derived"
    return None, "none"


birth_ranges = {}           # person id → (first day, last day) of the estimated birth date
birth_sources = Counter()   # where each birth range's date of death came from

for _person in people:
    if any(part is not None for part in carved_parts(_person, "birth")):
        continue
    if _person["age_of_death"] is None:
        continue
    _death, _source = death_span(_person)
    birth_sources[_source] += 1
    if _death is not None:
        birth_ranges[_person["id"]] = birth_range(_death, _person["age_of_death"])

_new_years = Counter(last.year - first.year for first, last in birth_ranges.values())
_decades = sum(first.year // 10 != last.year // 10 for first, last in birth_ranges.values())

print(f"{RULE}\nBirth ranges from the age at death\n{RULE}")
print(f"  {'Starting from':<40}{'people':>8}")
print(f"    {'a carved date of death':<38}{birth_sources['carved']:>8}")
print(f"    {'a date of death worked out above':<38}{birth_sources['derived']:>8}")
print(f"    {'[WARN] no date of death: no range':<38}{birth_sources['none']:>8}")
print(f"  {'Ranges reaching across':<40}")
for _crossings in sorted(_new_years):
    print(f"    {f'{_crossings} civil New Year(s)':<38}{_new_years[_crossings]:>8}")
print(f"    {'a decade':<38}{_decades:>8}")
print(RULE)
if birth_ranges:
    print(f"  {len(birth_ranges)} birth dates estimated, from "
          f"{min(first for first, _ in birth_ranges.values())} to "
          f"{max(last for _, last in birth_ranges.values())}")

────────────────────────────────────────────────────────────────────────
Birth ranges from the age at death
────────────────────────────────────────────────────────────────────────
  Starting from                             people
    a carved date of death                     369
    a date of death worked out above            36
    [WARN] no date of death: no range            4
  Ranges reaching across                  
    1 civil New Year(s)                        402
    2 civil New Year(s)                          3
    a decade                                    34
────────────────────────────────────────────────────────────────────────
  405 birth dates estimated, from 1830-09-08 to 1955-02-07


**5. Cross-check Against Carved Dates**

This cell checks the rules against the stones that carve both versions of a date. It takes `people` and:

1. converts the Hebrew date of death of every stone that carves both full dates of death with pyluach, and compares it with the carved civil date, counting the days between them
2. estimates the birth range of every stone that carves both a birth year and an age at death, and checks that the carved birth date lies within it

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-check Against Carved Dates  —  the rules on dates carved twice
#
# A stone that carves both full dates of death tests the conversion: the
# Hebrew day starts at nightfall, so a death in the evening is carved with a
# Hebrew date that converts to the next civil day, and a gap of weeks points
# to a misread day or month in the transcription rather than to the rule.
# A stone that carves a birth year and an age tests the birth formula.
# ─────────────────────────────────────────────────────────────────────────────
if "birth_ranges" not in globals():
    raise RuntimeError("Run the 'Derive the Birth Ranges' cell first.")

GAP_BANDS = ("Same day", "Conversion one day later (nightfall)", "Conversion one day earlier",
             "2 or 3 days apart", "4 to 45 days apart", "More than 45 days apart")


def gap_band(gap: int) -> str:
    """Return the GAP_BANDS label of a gap in days: the carved civil day less the converted one."""
    if gap == 0:
        return "Same day"
    if gap == -1:
        return "Conversion one day later (nightfall)"
    if gap == 1:
        return "Conversion one day earlier"
    if abs(gap) <= 3:
        return "2 or 3 days apart"
    return "4 to 45 days apart" if abs(gap) <= 45 else "More than 45 days apart"


def conversion_gap(person: dict) -> tuple[int, bool] | None:
    """Return the carved civil day minus the converted Hebrew day, and whether the years agree.

    None unless both dates of death are carved in full and both exist.
    """
    civil, hebrew = carved_parts(person, "death"), tuple(person[c] for c in HEBREW)
    if None in civil or None in hebrew:
        return None
    try:
        converted = dates.HebrewDate(hebrew[2], hebrew[1], hebrew[0]).to_pydate()
        carved = date(civil[2], civil[1], civil[0])
    except ValueError:
        return None
    return (carved - converted).days, carved.year == converted.year


gap_counts, same_year = Counter(), 0
for _person in people:
    _gap = conversion_gap(_person)
    if _gap is not None:
        gap_counts[gap_band(_gap[0])] += 1
        same_year += _gap[1]
compared = sum(gap_counts.values())

print(f"{RULE}\nConverted Hebrew date against the carved civil date\n{RULE}")
print(f"  {'Stones carving both full dates of death':<44}{'stones':>8}{'share':>9}")
for _label in GAP_BANDS:
    print(f"    {_label:<42}{gap_counts[_label]:>8}{gap_counts[_label] / compared:>9.1%}")
print(f"{RULE}\n  {same_year} of {compared} conversions land in the carved civil year "
      f"({same_year / compared:.1%})")

print(f"\n{RULE}\nCarved birth dates against the estimated range\n{RULE}")
birth_checks = []
for _person in people:
    _birth = carved_parts(_person, "birth")
    _death, _ = death_span(_person)
    if _birth[2] is None or _person["age_of_death"] is None or _death is None:
        continue
    _estimate, _carved = birth_range(_death, _person["age_of_death"]), civil_range(*_birth)
    _inside = _carved is not None and _estimate[0] <= _carved[1] and _carved[0] <= _estimate[1]
    birth_checks.append(_inside)
    _shown = "/".join(str(part) for part in _birth if part is not None)
    print(f"  {'OK ' if _inside else '!! '}{_person['ref']:<10} carved {_shown:<12} "
          f"age {_person['age_of_death']:>3}  range {_estimate[0]} to {_estimate[1]}")
print(f"{RULE}\n  {sum(birth_checks)} of {len(birth_checks)} carved birth dates "
      "lie within their estimated range")

────────────────────────────────────────────────────────────────────────
Converted Hebrew date against the carved civil date
────────────────────────────────────────────────────────────────────────
  Stones carving both full dates of death       stones    share
    Same day                                       283    70.6%
    Conversion one day later (nightfall)            32     8.0%
    Conversion one day earlier                      19     4.7%
    2 or 3 days apart                               16     4.0%
    4 to 45 days apart                              44    11.0%
    More than 45 days apart                          7     1.7%
────────────────────────────────────────────────────────────────────────
  394 of 401 conversions land in the carved civil year (98.3%)

────────────────────────────────────────────────────────────────────────
Carved birth dates against the estimated range
────────────────────────────────────────────────────────────────────────
  OK C-1-1      carved 1

**6. Write the Ranges**

This cell writes `death_ranges` and `birth_ranges` into `person` in one transaction, at the end of the run. It first empties every date derived before (both ends of each row whose upper year is set, so that only carved values remain), then fills each derived date's six columns. It commits when `APPLY` is True and rolls back otherwise, like `import.py --dry-run`. It reports:

1. how many rows it emptied and how many dates it wrote, for each kind of date
2. how many people's stored dates end up different from before the run

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Write the Ranges  —  one transaction: empty the old ranges, write the new
#
# Emptying first makes the write exact: a person whose range no longer
# applies (a corrected CSV, say) loses it, and a second run writes the same
# values over the same rows. Only the date columns of DATE_COLUMNS are
# written, and only for people of this cemetery's person table.
#   • APPLY = True   → committed: the API shows the ranges once restarted
#   • APPLY = False  → rolled back: the counts show what would change
# ─────────────────────────────────────────────────────────────────────────────
if "birth_ranges" not in globals():
    raise RuntimeError("Run the 'Derive the Birth Ranges' cell first.")


def clear_sql(kind: str) -> str:
    """Return the UPDATE that empties both ends of every derived `kind` date."""
    columns = ", ".join(f"{column} = NULL" for column in LOWER[kind] + UPPER[kind])
    return f"UPDATE person SET {columns} WHERE {UPPER[kind][2]} IS NOT NULL"


def write_sql(kind: str) -> str:
    """Return the UPDATE that fills the six columns of one person's derived `kind` date."""
    columns = ", ".join(f"{column} = %({column})s" for column in LOWER[kind] + UPPER[kind])
    return f"UPDATE person SET {columns} WHERE id = %(id)s"


def expected_dates(person: dict) -> dict:
    """Return the date columns a person's row should hold after the write."""
    expected = {column: None for column in DATE_COLUMNS}
    for kind, ranges in (("death", death_ranges), ("birth", birth_ranges)):
        if person["id"] in ranges:
            expected |= range_columns(kind, ranges[person["id"]])
        else:
            expected |= dict(zip(LOWER[kind], carved_parts(person, kind)))
    return expected


write_report = {}
with connect() as _conn:
    for _kind, _ranges in (("death", death_ranges), ("birth", birth_ranges)):
        _cleared = _conn.execute(clear_sql(_kind)).rowcount
        with _conn.cursor() as _cursor:
            _cursor.executemany(write_sql(_kind), [range_columns(_kind, span) | {"id": key}
                                                   for key, span in _ranges.items()])
        _stored = _conn.execute(f"SELECT count(*) AS n FROM person "
                                f"WHERE {UPPER[_kind][2]} IS NOT NULL").fetchone()["n"]
        write_report[_kind] = (_cleared, len(_ranges), _stored)
    changed = sum(any(_person[column] != value for column, value in expected_dates(_person).items())
                  for _person in people)
    if APPLY:
        _conn.commit()
    else:
        _conn.rollback()

print(f"{RULE}\nWrite to person in {describe_database(database_url())}"
      f"{'' if APPLY else '  (dry run: rolled back)'}\n{RULE}")
print(f"  {'Date':<10}{'emptied':>10}{'written':>10}{'stored after':>15}")
for _kind, (_cleared, _written, _stored) in write_report.items():
    print(f"  {'OK ' if _stored == _written else '!! '}{_kind:<7}{_cleared:>10}"
          f"{_written:>10}{_stored:>15}")
print(f"{RULE}\n  {changed} of {len(people)} people's dates "
      + ("changed" if APPLY else "would change") + " in the database")

────────────────────────────────────────────────────────────────────────
Write to person in neondb on ….neon.tech
────────────────────────────────────────────────────────────────────────
  Date         emptied   written   stored after
  OK death           0        90             90
  OK birth           0       405            405
────────────────────────────────────────────────────────────────────────
  459 of 703 people's dates changed in the database


**7. Consistency Check**

This cell reads `person` again and rebuilds every range a second way, from pyluach's first and last day of each Hebrew month or year instead of a list of days, and checks each birth range against the definition of an age in completed years. It checks that:

1. the second method gives exactly the ranges of sections 3 and 4
2. every row holds the dates it should: with `APPLY` the derived ranges and the carved values, and on a dry run the dates read in section 2
3. no other column of `person` and no other table changed

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency Check  —  every range rebuilt a second way, then compared
#
# The second method never lists the days of a year:
#   • a Hebrew date is bounded by month lengths: day d of month m is one
#     day; a month runs from its 1st to its length; a year from 1 Tishrei
#     to the day before the next 1 Tishrei; a day without a month from its
#     first month with that day to its last
#   • a birth range is checked by its definition: born on its first day,
#     the person is `age` on the first possible day of death and a day
#     earlier would make them older; born on its last, they are `age` on
#     the last possible day of death and a day later would make them younger
# ─────────────────────────────────────────────────────────────────────────────
if "write_report" not in globals():
    raise RuntimeError("Run the 'Write the Ranges' cell first.")


def month_bounds(year: int, month: int, day: int | None) -> tuple[date, date] | None:
    """Return the days of one Hebrew month that a date with `day` (or any day) can be."""
    length = len(hebrewcal.Month(year, month))
    if day is not None and day > length:
        return None
    first, last = (day, day) if day is not None else (1, length)
    return (dates.HebrewDate(year, month, first).to_pydate(),
            dates.HebrewDate(year, month, last).to_pydate())


def rebuild_hebrew(day: int | None, month: int | None,
                   year: int) -> tuple[date, date] | None:
    """Return a Hebrew date's first and last civil day, from month and year bounds."""
    if month is not None:
        if month == 13 and not hebrewcal.Year(year).leap:
            return None
        return month_bounds(year, month, day)
    if day is None:
        next_year = dates.HebrewDate(year + 1, 7, 1).to_pydate()
        return dates.HebrewDate(year, 7, 1).to_pydate(), next_year - timedelta(days=1)
    spans = [span for each in hebrewcal.Year(year).itermonths()
             if (span := month_bounds(year, each.month, day)) is not None]
    return (spans[0][0], spans[-1][1]) if spans else None


def age_on(birth: date, death: date) -> int:
    """Return the age in completed years on `death` of someone born on `birth`."""
    return death.year - birth.year - ((death.month, death.day) < (birth.month, birth.day))


def birth_fits(birth: tuple[date, date], death: tuple[date, date], age: int) -> bool:
    """True if `birth` is exactly the range of birth days that `death` and `age` allow."""
    one_day = timedelta(days=1)
    return (age_on(birth[0], death[0]) == age and age_on(birth[0] - one_day, death[0]) == age + 1
            and age_on(birth[1], death[1]) == age
            and age_on(birth[1] + one_day, death[1]) == age - 1)


with connect() as _conn:
    stored_people = {row["id"]: row for row in read_people(_conn)}
    tables_after = count_tables(_conn)

_rebuilt = {key: rebuild_hebrew(*(people_by_id[key][column] for column in HEBREW))
            for key in death_ranges}
death_mismatch = [key for key, span in death_ranges.items() if _rebuilt[key] != span]
birth_mismatch = [_person["ref"] for _person in people if _person["id"] in birth_ranges
                  and not birth_fits(birth_ranges[_person["id"]],
                                     _rebuilt.get(_person["id"]) or death_span(_person)[0],
                                     _person["age_of_death"])]

_checks = []
for _person in people:
    _row = stored_people.get(_person["id"])
    _want = expected_dates(_person) if APPLY else {c: _person[c] for c in DATE_COLUMNS}
    _dates_ok = _row is not None and all(_row[c] == v for c, v in _want.items())
    _others_ok = _row is not None and all(_row[c] == v for c, v in _person.items()
                                          if c not in DATE_COLUMNS)
    _checks.append((_person["ref"], _dates_ok, _others_ok))

_results = [
    ("death ranges rebuilt the second way", len(death_ranges) - len(death_mismatch),
     len(death_ranges), death_mismatch),
    ("birth ranges true to the age at death", len(birth_ranges) - len(birth_mismatch),
     len(birth_ranges), birth_mismatch),
    ("rows holding the expected dates", sum(ok for _, ok, _ in _checks), len(people),
     [ref for ref, ok, _ in _checks if not ok]),
    ("rows with other columns unchanged", sum(ok for _, _, ok in _checks), len(people),
     [ref for ref, _, ok in _checks if not ok]),
    ("other tables unchanged", sum(tables_after[t] == tables_before[t] for t in OTHER_TABLES),
     len(OTHER_TABLES), [t for t in OTHER_TABLES if tables_after[t] != tables_before[t]]),
]
print(f"{RULE}\nConsistency: person in {describe_database(database_url())} vs the ranges "
      f"derived above\n{RULE}")
for _label, _passed, _total, _problems in _results:
    print(f"  {'OK ' if not _problems else '!! '}{_label:<40}{_passed:>5} of {_total}")
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
consistent = all(not problems for _, _, _, problems in _results)
print(f"{RULE}\n  {len(stored_people)} of {len(people)} people checked  "
      f"({'consistent' if consistent else 'see the problems above'})")

print(f"\n✓ Date ranges complete ({len(death_ranges)} dates of death and {len(birth_ranges)} "
      f"birth dates {'written' if APPLY else 'derived; dry run, nothing written'}).")
print(f"  consistent: {consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: person in neondb on ….neon.tech vs the ranges derived above
────────────────────────────────────────────────────────────────────────
  OK death ranges rebuilt the second way        90 of 90
  OK birth ranges true to the age at death     405 of 405
  OK rows holding the expected dates           703 of 703
  OK rows with other columns unchanged         703 of 703
  OK other tables unchanged                      4 of 4
────────────────────────────────────────────────────────────────────────
  703 of 703 people checked  (consistent)

✓ Date ranges complete (90 dates of death and 405 birth dates written).
  consistent: True
